# Track B — Qwen3-VL-8B (Colab A100) — **_0716 No_ordering 재해석 라운드**

기준선: 8B + `mid` + TTA8 = **LB 0.88** (_0712). 이 라운드는 **데이터 레시피만** 바꾼다.

## 발견 (사용자, 2026-07-16)
공식 정의 **"No_ordering=True면 이미지는 셔플링되지 않았으며 정답은 [1,2,3,4] 고정"** — 즉
*생성 과정*(안 섞음)이지 *콘텐츠 속성*(정렬 불가)이 아니다. no_ordering 샘플은 **정상 영상**이다.
- 결정타: `em_no_ordering` 0.596 > `em_orderable` 0.5025 — TTA가 셔플한 원순서를 59.6% 복원(정렬 불가면 불가능)
- 실제 버그 둘: ① 증강이 15.5%에 거짓 라벨(섞고도 identity 고정) ② `mid`가 없는 개념(UNORDERABLE)을 가르침
- 상세: `src/train/targets.py` docstring / `reports/preprocessing.md` §B3-3

## 이번 라운드 = `plain` 스타일
- UNORDERABLE 개념 제거 → **순수 24-way 순서 맞추기**. 게이트·센티넬·이진분류 소거.
- 증강은 `augment_perm_truthful`(라벨 항상 진실) + `identity_prior=0.155`(test 사전확률).
- 하드 반복은 **Phase 1에서 off** (`hard_aug_max_repeats=1`) — 전달본 repeats는 구 모델
  tta1+게이트off 채굴 아티팩트라 근거 없음. 진짜 부스팅은 **Phase 1.5**(H1'로 재채굴).

## 실행 순서
`A1→A2→A3→A4`(재시작)`→A1,A3` → `B0`(스키마) → `B1`(스모크)`→B2`(학습)`→B3`(병합)
→ **`B5`(재시작)**`→A1,A3` → `B4`(병합게이트) → **`C0`(기준선)** → `C1`(val 게이트) → `C2`(제출)
→ (게이트 통과 시) `H1'`(재채굴) → 로컬 재학습(Phase 1.5)

## 제출 규율
판정자는 **C1 val EM ≥ C0 기준선**(기존 8B mid). 못 넘으면 제출 않고 델타(레시피/증강본) 분리.
다중 델타(8B는 그대로·레시피·새 증강본)이므로 C0 없이는 해석 불가 — **C0 필수**.

## 사전 준비 (Drive `MyDrive/snuai/`)
- `snuai_code_0716.zip` (로컬 `python -m cloud.pack_code` → `outputs/snuai_code.zip` 개명)
- `access_token`, 기존 8B 병합모델(C0 기준선용, 경로는 C0 셀 `BASE_8B`에 기입)


In [ ]:
# A1) Drive 마운트 + 코드 압축 해제 (기존 코드 청소 후 — 구버전 파일 잔류 방지)
# 필수: 로컬 outputs/snuai_code.zip 을 snuai_code_0716.zip 으로 개명해 Drive MyDrive/snuai/ 에 업로드.
from google.colab import drive
drive.mount('/content/drive')

import os
ZIP = '/content/drive/MyDrive/snuai/snuai_code_0716.zip'
assert os.path.exists(ZIP), f'코드 zip 없음: {ZIP} — 로컬 outputs/snuai_code.zip을 개명해 업로드하세요'
!rm -rf /content/code && unzip -qo {ZIP} -d /content/code
# 압축 해제 검증 — 조용한 실패 방지
for must in ('/content/code/src/infer/predict.py', '/content/code/outputs/split.csv',
             '/content/code/outputs/sft_train_llm_aug_hard_nogate.jsonl',  # _0716 학습 데이터
             '/content/code/configs/sft_qwen8b.yaml'):
    assert os.path.exists(must), f'압축 해제 불완전: {must} 없음 — zip 내용을 확인하세요'
# _0716 신선도: plain 스타일이 실제로 풀렸는지 확인. 구 zip이면 B1이 "unknown style"로 뒤늦게 터진다.
import sys; sys.path.insert(0, '/content/code')
from src.train.targets import STYLES, augment_perm_truthful  # noqa
assert 'plain' in STYLES, 'zip이 구버전 — outputs/snuai_code.zip을 다시 말아 _0716으로 업로드하세요 (plain 없음)'
print('code ready:', ZIP, '| STYLES:', STYLES)


In [ ]:
# A2) 대회 데이터 — Kaggle API v2 인증 (단일 토큰 방식)
# 토큰 발급: kaggle.com/settings → API → "Generate New Token" → 화면에 뜬 문자열 복사
SLUG = 'snuaichallenge'
!pip install -qU kaggle  # v2 CLI 보장 (access_token 지원)
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/snuai/access_token ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
!kaggle competitions download {SLUG} -p /content
!unzip -qo /content/{SLUG}.zip -d /content/data
!ls /content/data


In [ ]:
# A3) 경로 설정 — train.csv 위치 자동 탐지 (중첩 폴더 대응)
import os, glob, shutil
hits = sorted(glob.glob('/content/data/train.csv')
              + glob.glob('/content/data/*/train.csv')
              + glob.glob('/content/data/*/*/train.csv'))
assert hits, 'train.csv를 찾지 못함 — A2의 압축 해제 결과 확인'
DATA_ROOT = os.path.dirname(hits[0])
print('DATA_ROOT:', DATA_ROOT)

yaml_text = '\n'.join([
    f'data_dir: {DATA_ROOT}',
    f'train_csv: {DATA_ROOT}/train.csv',
    f'test_csv: {DATA_ROOT}/test.csv',
    f'sample_submission: {DATA_ROOT}/sample_submission.csv',
    f'train_image_dir: {DATA_ROOT}/train',
    f'test_image_dir: {DATA_ROOT}/test',
    'models_dir: /content/models',
    'outputs_dir: /content/outputs',
    'reports_dir: /content/reports',
])
open('/content/paths.yaml', 'w').write(yaml_text)
os.environ['SNUAI_PATHS_CONFIG'] = '/content/paths.yaml'
os.makedirs('/content/outputs', exist_ok=True)
# split.csv는 코드 번들에 있으므로 outputs_dir로 복사 (--fold val 경로가 참조)
shutil.copy('/content/code/outputs/split.csv', '/content/outputs/split.csv')
# 주의: _0716 plain은 hard_cases_path를 쓰지 않는다 (전달본 jsonl에 재가중 필드가 병합돼 있음).
# 따라서 hard_train_cases.csv 복사는 불필요 — 대신 재채굴(H1')이 outputs/에 새로 쓴다.

import sys; sys.path.insert(0, '/content/code')
from src.data.loader import load_split
print('train rows:', len(load_split('train')), '| test rows:', len(load_split('test')))


In [ ]:
# A4) 설치 + 재현성 증빙
# ⚠️ 이 셀을 처음 실행한 뒤에는 반드시 [런타임 → 세션 다시 시작] 후 A1, A3만 재실행하고 B0으로.
#    (unsloth가 pyarrow 등을 교체하므로 재시작 없이는 바이너리 불일치 에러 발생)
!pip install -q unsloth imagehash
import subprocess, sys
open('/content/drive/MyDrive/snuai/pip_freeze_colab.txt','w').write(
    subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True).stdout)
print('설치 완료 — 런타임을 재시작한 뒤 A1, A3 재실행 후 B0을 진행하세요.')


In [ ]:
# B0) 전달 학습 데이터 스키마 검증 — plain 진실 라벨 전제 확인 (학습 전 필수)
# plain 증강은 "no_ordering 레코드의 rank가 이미 [1,2,3,4](진실)"임을 전제로 한다.
# 이게 깨지면 거짓 라벨을 다시 학습하므로 여기서 즉시 잡는다.
import json
SFT = '/content/code/outputs/sft_train_llm_aug_hard_nogate.jsonl'
recs = [json.loads(l) for l in open(SFT, encoding='utf-8')]
n = len(recs); n_no = sum(bool(r['no_ordering']) for r in recs)
bad = [r['Id'] for r in recs if r['no_ordering'] and r['rank'] != [1, 2, 3, 4]]
has_llm = sum(bool(r.get('caption_llm_variants')) for r in recs)
print(f'records={n}, no_ordering={n_no} ({n_no/n:.4f}), caption_llm_variants={has_llm}')
assert not bad, f'no_ordering인데 rank!=[1,2,3,4]인 레코드 {len(bad)}건: {bad[:5]} — 진실 증강 전제 위반'
assert abs(n_no/n - 0.155) < 0.01, f'no_ordering 비율 {n_no/n:.4f}가 기저율 0.155에서 벗어남'
print('스키마 검증 통과 — B1 진행 (rank 진실성 OK, hard 반복은 config에서 off)')


In [ ]:
# B1) 스모크 (32샘플) — 장기 학습 전 필수. 8B는 A100/L4 전용 (T4 OOM)
import sys
sys.path.insert(0, '/content/code')  # 재실행 대비
from cloud.train_unsloth import run
SFT = '/content/code/outputs/sft_train_llm_aug_hard_nogate.jsonl'  # _0716 팀원 LLM 증강본
CFG = '/content/code/configs/sft_qwen8b.yaml'                       # style: plain, identity_prior 0.155
OUT = '/content/drive/MyDrive/snuai/outputs/qwen3vl8b_0716'         # Drive = 세션 휘발 대비
run(CFG, SFT, DATA_ROOT, limit=32, output_dir=OUT + '_smoke')


In [ ]:
# B2) 본 학습 (_0716: 8B + plain + 진실증강 + 하드반복 off — configs/sft_qwen8b.yaml)
# 유효 데이터 8,582행 (하드 반복 off라 확장 없음 — no_ordering 15.5% = test 기저율).
# 첫 실행 resume=False, 세션 끊긴 뒤 재실행은 resume=True.
# 8B 배치 (유효 배치 16 고정): A100 40GB는 {2, 8}로 시작(OOM 시 {1, 16}로 낮춤).
#   8B는 7B보다 vision forward가 무거우므로 여유 없으면 per_device_batch 1이 안전.
OVERRIDES = {'per_device_batch': 2, 'grad_accum': 8}  # A100. OOM 시 {'per_device_batch':1,'grad_accum':16}
lora_dir = run(CFG, SFT, DATA_ROOT, resume=False, output_dir=OUT, train_overrides=OVERRIDES)
print('LoRA saved:', lora_dir)


In [ ]:
# B3) 병합 저장 (Drive) — 추론은 플레인 transformers 경로 사용
# ⚠️ save_pretrained_merged는 vision 모델에서 LoRA를 병합하지 않고 베이스만 저장
#    (unsloth#1352) → peft 표준 병합 사용.
from unsloth import FastVisionModel
OUT = '/content/drive/MyDrive/snuai/outputs/qwen3vl8b_0716'
MERGED = '/content/drive/MyDrive/snuai/qwen3vl8b_merged_0716'
model, processor = FastVisionModel.from_pretrained(OUT + '/lora', load_in_4bit=False)
merged = model.merge_and_unload()
merged.save_pretrained(MERGED)
processor.save_pretrained(MERGED)
print('merged saved:', MERGED, '— B5로 런타임 재시작 후 B4 병합 검증을 통과해야 추론 가능')


In [ ]:
# B5) ⚠️ 추론 전 런타임 재시작 (8B OOM 방지 — _0711 실측 교훈)
# B3 병합 셀이 노트북 커널 GPU에 bf16 8B(~20GB)를 잔류시킨다. 이 상태로 C0/C1 추론(배치 16 =
# 64장/vision forward)을 돌리면 A100 40GB를 초과해 CUDA OOM이 난다.
# → [런타임 → 세션 다시 시작] 후 A1, A3만 재실행하고 B4부터 이어갈 것.
#   (병합모델은 Drive에 있으므로 재학습 불요. 아래 assert로 재시작 여부를 강제 확인)
import torch, sys
assert 'FastVisionModel' not in dir(), 'B3의 모델이 아직 커널에 있음 — 런타임을 재시작하세요'
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'GPU free {free/1e9:.1f}GB / {total/1e9:.1f}GB')
    assert free > total * 0.7, 'GPU 메모리가 비어있지 않음 — 런타임 재시작 후 A1,A3만 재실행하고 여기로'
print('GPU 클린 — B4 진행 가능')


In [ ]:
# B4) 병합 검증 게이트 (필수) — train 8샘플 생성으로 LoRA가 실제 병합됐는지 확인
# 병합 실패 시(베이스만 저장됨) 정답률이 랜덤(~0/8). 정상 기준: 4/8 이상. **--style plain**.
MERGED = '/content/drive/MyDrive/snuai/qwen3vl8b_merged_0716'
!rm -f /content/outputs/merge_check.jsonl
!cd /content/code && python -m src.infer.predict --model {MERGED} --split train --fold train \
    --limit 8 --tta 1 --batch 8 --style plain --out /content/outputs/merge_check.jsonl
import sys, json; sys.path.insert(0, '/content/code')
from src.utils.permutation import parse_permutation, unshuffle_rank_label, parse_answer_column
from src.data.loader import load_split
truth = load_split('train').set_index('Id')
hits = n = 0
for r in map(json.loads, open('/content/outputs/merge_check.jsonl')):
    rank = parse_permutation(r['text']); n += 1
    hits += rank is not None and unshuffle_rank_label(rank, r['perm']) == parse_answer_column(truth.loc[r['Id'], 'Answer'])
print(f'merge check: {hits}/{n} 일치')
assert n >= 8 and hits >= 4, '병합 실패 의심 (unsloth#1352) — B3 재확인 전 추론 진행 금지'
print('병합 검증 통과 — C0 진행')


In [ ]:
# C0) 기준선 EM — 기존 8B(mid, LB 0.88)의 val EM. **다중 델타의 유일한 판정자.**
# 이 라운드는 8B는 그대로 두고 레시피(plain)+증강본을 동시에 바꾼다. C1이 이겼는지 판정하려면
# "같은 8B가 구 레시피로 낸 val EM"이 필요하다.
#  - 기존 raw_val이 Drive에 있으면 재생성 없이 그걸로 채점 (무료·즉시).
#  - 없으면 기존 8B 병합모델로 val tta8 1회 생성 (~40분). BASE_8B에 경로를 기입할 것.
import os
BASE_RAW = '/content/drive/MyDrive/snuai/raw_val_8b_mid_baseline.jsonl'  # 기존 산출물 있으면 여기로
BASE_8B  = '/content/drive/MyDrive/snuai/qwen3vl8b_merged_0712'          # ← 기존 8B 병합모델 경로 확인·수정
if not os.path.exists(BASE_RAW):
    assert os.path.exists(BASE_8B), f'기존 8B 모델 없음: {BASE_8B} — 실제 경로로 수정하세요'
    !cd /content/code && python -m src.infer.predict --model {BASE_8B} --split train --fold val \
        --style mid --tta 8 --batch 16 --out {BASE_RAW}
!cd /content/code && python -m src.infer.aggregate --raw {BASE_RAW} --out /content/outputs/pred_val_base.csv
print('=== 기준선 (8B mid) ===')
!cd /content/code && python -m src.eval.em --pred /content/outputs/pred_val_base.csv --fold val


In [ ]:
# C1) val 게이트 (제출 전 필수) — _0716 plain 모델. raw는 Drive에 직접 저장.
# 통과 조건 3가지 (셋 다 봐야 함):
#   ① EM ≥ C0 기준선 (8B mid의 val EM)
#   ② identity_rate ≈ 0.155 (sanity — plain 사전확률이 test와 정합하는가)
#   ③ em_no_ordering·em_orderable 양쪽 개선 방향 (레시피가 맞다면 no_ordering↑, orderable 비하락)
MODEL = '/content/drive/MyDrive/snuai/qwen3vl8b_merged_0716'
!cd /content/code && python -m src.infer.predict --model {MODEL} --split train --fold val \
    --style plain --tta 8 --batch 16 --out /content/drive/MyDrive/snuai/raw_val_0716.jsonl
!cd /content/code && python -m src.infer.aggregate --raw /content/drive/MyDrive/snuai/raw_val_0716.jsonl \
    --out /content/outputs/pred_val.csv
!cd /content/code && python -m src.eval.em --pred /content/outputs/pred_val.csv --fold val
print('→ 위 EM을 C0 기준선과 비교. 못 넘으면 제출 중단하고 원인(레시피 vs 증강본) 분리 실험.')


In [ ]:
# C2) test 추론 → 검증된 submission (C1이 기준선을 넘었을 때만) — raw는 Drive에 직접 저장.
# 손상 이미지로 중단되면 데이터 재압축해제 후 같은 명령 재실행 (완료분 자동 스킵).
MODEL = '/content/drive/MyDrive/snuai/qwen3vl8b_merged_0716'
!cd /content/code && python -m src.infer.predict --model {MODEL} --split test \
    --style plain --tta 8 --batch 16 --out /content/drive/MyDrive/snuai/raw_test_0716.jsonl
!cd /content/code && python -m src.infer.aggregate --raw /content/drive/MyDrive/snuai/raw_test_0716.jsonl \
    --submission /content/drive/MyDrive/snuai/submission_0716.csv
print('Drive의 snuai/submission_0716.csv를 다운로드해 Kaggle에 제출')


## Phase 1.5 — 진짜 부스팅: plain 모델 **자신의 오답**으로 재채굴

전달본의 `caption_aug_repeats`는 구 모델(7B mid, tta1+게이트off) 채굴 아티팩트라 Phase 1에서
껐다. 부스팅의 원리는 "**직전 모델**의 오답에 가중"이므로, C1 게이트를 통과한 plain 모델로
train fold를 다시 채점해 오답노트를 새로 만든다. **C1 통과 후에만** 실행한다.

`H1'`(아래, ~4h A100) → 로컬에서 `hard_cases.py`로 재가중 CSV 생성 → `sft_qwen8b_v2.yaml`
(hard_cases_path 지정 또는 jsonl 재병합) → B1~C2 재실행.

In [ ]:
# H1') plain 모델로 train fold 재채굴 → 하드 마이닝 입력 (Phase 1.5)
# tta4 + 게이트 기본값이 tta1보다 정확 (predict는 resume이라 같은 raw에 뷰 추가 가능).
# 규모: train fold 8,582 × tta4 = 3.4만 생성 (~4h A100). --style plain 필수.
MODEL = '/content/drive/MyDrive/snuai/qwen3vl8b_merged_0716'
!cd /content/code && python -m src.infer.predict --model {MODEL} --split train --fold train \
    --style plain --tta 4 --batch 16 --out /content/drive/MyDrive/snuai/raw_trainfold_0716.jsonl
# plain은 UNORDERABLE이 없어 disperse_gate 기본값(on) 그대로 — tta4면 정상 동작.
!cd /content/code && python -m src.infer.aggregate --raw /content/drive/MyDrive/snuai/raw_trainfold_0716.jsonl \
    --out /content/drive/MyDrive/snuai/pred_trainfold_0716.csv
print('pred_trainfold_0716.csv를 다운로드해 로컬에서:')
print('  python -m src.preprocess.hard_cases --pred pred_trainfold_0716.csv --fold train \\')
print('      --out outputs/hard_train_cases_0716.csv')
print('  # 이어서 llm_caption_augment로 재증강 후 sft_qwen8b_v2.yaml로 Phase 1.5 재학습')


## Phase 2 — GRPO (조건부, 별도 계획으로 착수)

**결론: DPO보다 GRPO** — EM을 직접 보상으로 쓰고(우도 마진 아님), 오답쌍 채굴이 불필요하며,
roadmap L1("EM 보상·단일 체크포인트")로 이미 계획돼 있다. SFT→GRPO는 동일 어댑터 연속 학습이라
**단일 체크포인트 유지**(앙상블 금지 ✓). 제공 데이터 + 라벨 유래 보상의 로컬 RL = 규정 적합.

**착수 조건(모두 충족 시만)**: ① Phase 1 val 게이트 통과 ② 마감까지 ≥2일 ③ Unsloth VLM GRPO
스모크 통과. 미충족이면 DPO 폴백 또는 Phase 1 결과로 종료.

**설계 스케치**: Phase-1 LoRA에서 이어서, 보상 = EM(1/0) + 파싱가능 보너스(소액), 프롬프트당
8 샘플·temperature, plain 타깃 ~30토큰이라 생성 비용 소형. 프롬프트 풀은 hard_score 상위 우선.
KL/entropy 페널티로 identity 붕괴 감시(출력 identity 비율 vs 0.155). — 구현은 별도 라운드.